# 06 NMI predictive linkage

This notebook owns the new NMI-facing computations directly, following the same pattern as notebooks 00--05. It uses the existing scientific models in `mrl_trace`, reads the measured Au and ITO files, and keeps every result in notebook variables unless result saving is explicitly enabled.

It does **not** replace the original full-sweep experiment record. The original downstream ladder retains its recorded parameter provenance; this notebook adds only the new held-out material-to-learning tests.


## Workload controls

Set `MRL_RUN_PROFILE=smoke|reduced|publication` and `MRL_WORKERS`. The default `reduced` execution is suitable for a laptop feasibility run. Smoke and reduced outputs are feasibility diagnostics; only `publication` is intended for final inference. Set `MRL_SAVE_RESULTS=1` and `MRL_OUTPUT_DIR` to write the result archive outside Git for later DOI deposition.

| Analysis | Smoke | Reduced | Publication |
|---|---:|---:|---:|
| Au/ITO bootstrap | 500 | 2,000 | 10,000 |
| Primary timing | 2 seeds x 80 trials | 6 x 500 | 20 x 1,500 |
| Supported-k sensitivity | 1 x 30 | 3 x 300 | 20 x 1,500 |
| Retention holdout | 2 x 80 episodes | 6 x 300 | 20 x 800 |
| Reversal quantiles | 3 x 400 trials | 6 x 1,200 | 20 x 3,000 |

Laptop smoke run (PowerShell):

```powershell
python -m pip install -e ".[repro,test]"
$env:MRL_RUN_PROFILE = "smoke"
$env:MRL_WORKERS = "4"
python -m jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --output 06_nmi_predictive_linkage.executed.ipynb experiments/06_nmi_predictive_linkage.ipynb
```

For a publication run, change the profile to `publication`, set `MRL_SAVE_RESULTS=1`, and set `MRL_OUTPUT_DIR` to persistent storage outside the checkout. The same command runs on a workstation, HPC node, ordinary cloud VM or SageMaker; SageMaker is not scientifically required. Start with smoke and reduced runs to measure scaling before selecting hardware. Run the publication profile twice into distinct directories and archive the executed notebooks, NumPy/JSON outputs, commit, environment and logs with the DOI.

## Archive compatibility and provenance

The absence of full-sweep `.npy` files from Git is not evidence that the original experiments were not run. The original parameter and source-data tables record the trace, spiking, dense retention-delay, sequential T-maze, deep XOR/temporal-distractor, shallow-DMS and one-retention reversal results. Historical arrays remain recoverable from commit `2288c53`; most are reduced examples, while `exp6_sequential.npy` contains the full 20-seed, 800-episode result. Rerun an original downstream experiment only if the manuscript attributes it to the newly refitted physical law or requires a new paired per-seed comparison.

The genuinely new computations owned here are replicate-level held-out physical-model comparison, empirical-ITO timing prediction with matched controls, supported-cascade-depth sensitivity, predefined retention scaling with four empirical holdouts, and four-quantile reversal with a decay-matched exponential. Git retains code, reduced executable notebooks, parameter/source tables and small fixtures. Publication-scale per-seed outputs, logs, environment records and figure-source manifests belong in the archival DOI.


In [ ]:
from pathlib import Path
from IPython.display import display
import csv, hashlib, json, math, os, re, sys, time

import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd().resolve()
REPO_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "src" / "mrl_trace").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the mrl-trace checkout or a subdirectory")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

RUN_PROFILE = os.getenv("MRL_RUN_PROFILE", "reduced").strip().lower()
if RUN_PROFILE == "quick":
    RUN_PROFILE = "smoke"
if RUN_PROFILE not in {"smoke", "reduced", "publication"}:
    raise ValueError("MRL_RUN_PROFILE must be smoke, reduced or publication")
WORKERS = max(1, int(os.getenv("MRL_WORKERS", "1")))
SAVE_RESULTS = os.getenv("MRL_SAVE_RESULTS", "0").lower() in {"1", "true", "yes"}
OUTPUT_DIR = Path(os.getenv("MRL_OUTPUT_DIR", REPO_ROOT / "data" / "nmi-results")).expanduser().resolve()
BOOTSTRAP = {"smoke": 500, "reduced": 2000, "publication": 10000}[RUN_PROFILE]
PROVENANCE_RECORD = {
    "historical_archive_commit": "2288c53bde5fd3fbc41c4db43cd9c48a7ce5ab4a",
    "original_law": {"beta_fill": 2.0, "cascade_k": 3,
                     "tau_r_s": [145.0, 2.9], "tau_d_s": [11700.0, 3.9]},
    "original_full_sweeps_recorded": True,
    "new_computation": ["physical_model_comparison", "timing_prediction",
                         "supported_k_sensitivity", "retention_holdouts",
                         "reversal_quantiles"],
    "distribution": {"git": "code, reduced notebooks, tables, small fixtures",
                     "doi": "full per-seed outputs, logs, environment, figure sources"},
}
print({"profile": RUN_PROFILE, "workers": WORKERS, "bootstrap": BOOTSTRAP,
       "save_results": SAVE_RESULTS, "output_dir": str(OUTPUT_DIR)})


## 1. Replicate-level physical identification

The following cell contains the complete measured-data analysis used here: raw-file hashing, grouped leave-one-bias-out Au model comparison, explicit ITO quality control and experimental-trace bootstrap summaries. It is embedded so this experiment does not depend on a new source-package module.


In [ ]:
"""Publication-grade analysis of the measured Au and ITO transients.

The learning modules deliberately remain small numerical reference models.  This
workflow implements the stronger evidential contract needed by the paper: every raw file is
hashed, Au traces are fitted as individual replicates with bias-grouped validation,
and every ITO workbook receives an explicit quality-control outcome.  No synthetic
device population is created here.
"""
from __future__ import annotations

import contextlib
import csv
import hashlib
import io
import json
import math
import re
from pathlib import Path
from typing import Iterable

import numpy as np
from scipy.optimize import least_squares
from scipy.special import gammainc

TRAIN_BIASES = (0.8, 0.9, 1.1, 1.2, 1.4, 1.5)
OOD_BIASES = (1.7, 1.8)
GOLD_RE = re.compile(r"trace_V([mp])(\d+(?:\.\d+)?)_tr(\d+)\.csv$")
ITO_RE = re.compile(
    r"(?P<protocol>voltage bias|current stress)#\d+ Run(?P<run>\d+) "
    r"(?P<date>\d{2}-\d{2}-\d{4})\.xls$",
    re.IGNORECASE,
)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def resolve_gold_dir(value: str | Path | None = None) -> Path:
    """Resolve the Au CSV directory without creating it."""
    import os

    candidates = []
    if value is not None:
        candidates.append(Path(value))
    if os.environ.get("MRL_TRACE_GOLD_DIR"):
        candidates.append(Path(os.environ["MRL_TRACE_GOLD_DIR"]))
    root = REPO_ROOT
    candidates.extend((
        root / "data" / "raw" / "gold_export",
        root.parent / "current_transient" / "gold_export",
        root / "data" / "device_model" / "gold_export",
    ))
    for candidate in candidates:
        if candidate.is_dir() and len(list(candidate.glob("trace_*.csv"))) == 24:
            return candidate.resolve()
    raise FileNotFoundError(
        "Au raw directory with 24 trace CSVs not found; set MRL_TRACE_GOLD_DIR"
    )


def resolve_ito_dir(value: str | Path | None = None) -> Path:
    """Resolve the ITO workbook directory without creating it."""
    import os

    candidates = []
    if value is not None:
        candidates.append(Path(value))
    if os.environ.get("MRL_TRACE_ITO_DIR"):
        candidates.append(Path(os.environ["MRL_TRACE_ITO_DIR"]))
    root = REPO_ROOT
    candidates.extend((root / "data" / "raw" / "ito", Path.home() / "Downloads" / "ITO data"))
    for candidate in candidates:
        if candidate.is_dir() and len(list(candidate.glob("*.xls"))) == 91:
            return candidate.resolve()
    raise FileNotFoundError(
        "ITO raw directory with 91 .xls workbooks not found; set MRL_TRACE_ITO_DIR"
    )


def _read_xls(path: Path) -> tuple[list[str], np.ndarray, dict[str, str]]:
    try:
        import xlrd
    except ImportError as exc:  # pragma: no cover - dependency error is explicit
        raise RuntimeError("ITO parsing requires the 'repro' extra (xlrd)") from exc

    # The instrument emits harmless OLE consistency notices for its non-sector-aligned
    # legacy files. Keep them out of the publication log while preserving the raw hash.
    notices = io.StringIO()
    with contextlib.redirect_stdout(notices), contextlib.redirect_stderr(notices):
        book = xlrd.open_workbook(path, on_demand=True, logfile=notices)
    sheet = book.sheet_by_name("Sheet1")
    headers = [str(sheet.cell_value(0, col)).strip() for col in range(sheet.ncols)]
    values = np.asarray([
        [sheet.cell_value(row, col) for col in range(sheet.ncols)]
        for row in range(1, sheet.nrows)
    ], dtype=float)
    metadata: dict[str, str] = {}
    if "Sheet3" in book.sheet_names():
        meta = book.sheet_by_name("Sheet3")
        for row in range(meta.nrows):
            key = str(meta.cell_value(row, 0)).strip()
            if key:
                metadata[key] = str(meta.cell_value(row, 1)).strip()
    book.release_resources()
    return headers, values, metadata


def raw_manifest(gold_dir: Path, ito_dir: Path) -> list[dict[str, object]]:
    """Return one immutable provenance row for every raw source file."""
    rows: list[dict[str, object]] = []
    for path in sorted(gold_dir.glob("trace_*.csv")):
        match = GOLD_RE.match(path.name)
        if not match:
            continue
        sign, magnitude, trial = match.groups()
        with path.open("r", encoding="utf-8", newline="") as fh:
            reader = csv.reader(fh)
            columns = next(reader)
            sample_count = sum(1 for _ in reader)
        rows.append({
            "dataset": "au_transient", "filename": path.name,
            "sha256": sha256_file(path), "bytes": path.stat().st_size,
            "electrode": "Au/Ti", "protocol": "voltage_step",
            "bias_v": (-1 if sign == "m" else 1) * float(magnitude),
            "trial_or_run": int(trial), "date": "not_recorded",
            "columns": "|".join(columns), "sample_count": sample_count,
            "qc_status": "pending_fit", "qc_reason": "",
        })

    for path in sorted(ito_dir.glob("*.xls")):
        match = ITO_RE.search(path.name)
        headers, values, metadata = _read_xls(path)
        protocol = match.group("protocol").lower().replace(" ", "_") if match else "unknown"
        bias = metadata.get("Bias", "")
        try:
            bias_v: float | str = float(bias)
        except ValueError:
            bias_v = ""
        rows.append({
            "dataset": "ito_relaxation", "filename": path.name,
            "sha256": sha256_file(path), "bytes": path.stat().st_size,
            "electrode": "ITO", "protocol": protocol, "bias_v": bias_v,
            "trial_or_run": int(match.group("run")) if match else "",
            "date": match.group("date") if match else metadata.get("Last Executed", ""),
            "columns": "|".join(headers), "sample_count": int(values.shape[0]),
            "qc_status": "pending_fit" if protocol == "voltage_bias" else "not_decay_protocol",
            "qc_reason": "conditioning/current-stress file" if protocol != "voltage_bias" else "",
        })
    return rows


def load_gold_traces(gold_dir: Path, *, n_grid: int = 240) -> list[dict[str, object]]:
    traces: list[dict[str, object]] = []
    for path in sorted(gold_dir.glob("trace_*.csv")):
        match = GOLD_RE.match(path.name)
        if not match:
            continue
        sign, magnitude, trial = match.groups()
        arr = np.genfromtxt(path, delimiter=",", names=True)
        t = np.asarray(arr["time"], float)
        current = np.abs(np.asarray(arr["current"], float))
        valid = np.isfinite(t) & np.isfinite(current) & (t >= 0) & (current > 0)
        t, current = t[valid], current[valid]
        if t.size < 20:
            raise ValueError(f"too few valid Au samples in {path.name}")
        indices = np.unique(np.linspace(0, t.size - 1, min(n_grid, t.size)).astype(int))
        traces.append({
            "filename": path.name, "bias": float(magnitude), "signed_bias":
            (-1 if sign == "m" else 1) * float(magnitude), "trial": int(trial),
            "time": t[indices] - t[indices][0], "current": current[indices],
        })
    if len(traces) != 24:
        raise ValueError(f"expected 24 Au traces, found {len(traces)}")
    return traces


def _kinetic_shape(t: np.ndarray, bias: float, theta: np.ndarray,
                   model: str, k: int | None = None) -> np.ndarray:
    tr0, cr, td0, cd = math.exp(theta[0]), theta[1], math.exp(theta[2]), theta[3]
    tau_r = tr0 * math.exp(-cr * bias)
    tau_d = td0 * math.exp(-cd * bias)
    if model == "kww":
        beta = math.exp(theta[4])
        rise = 1.0 - np.exp(-np.power(np.maximum(t, 0) / tau_r, beta))
    elif model == "cascade":
        if k is None:
            raise ValueError("cascade model requires k")
        rise = gammainc(k, k * np.maximum(t, 0) / tau_r)
    else:
        raise ValueError(model)
    return rise * np.exp(-np.maximum(t, 0) / tau_d)


def _affine_fit(shape: np.ndarray, observed: np.ndarray) -> tuple[np.ndarray, float, float]:
    design = np.column_stack((shape, np.ones_like(shape)))
    coef, *_ = np.linalg.lstsq(design, observed, rcond=None)
    return design @ coef, float(coef[0]), float(coef[1])


def _fit_gold_model(traces: Iterable[dict[str, object]], model: str,
                    k: int | None = None) -> tuple[np.ndarray, list[dict[str, float]]]:
    traces = list(traces)
    initial = np.asarray([math.log(145.0), 2.9, math.log(11700.0), 3.9], float)
    lower = np.asarray([math.log(0.1), 0.0, math.log(1.0), 0.0], float)
    upper = np.asarray([math.log(1e5), 10.0, math.log(1e7), 10.0], float)
    if model == "kww":
        initial = np.r_[initial, math.log(2.0)]
        lower = np.r_[lower, math.log(0.4)]
        upper = np.r_[upper, math.log(4.0)]

    def residual(theta: np.ndarray) -> np.ndarray:
        chunks = []
        for trace in traces:
            y = np.asarray(trace["current"], float)
            shape = _kinetic_shape(np.asarray(trace["time"], float), float(trace["bias"]),
                                   theta, model, k)
            prediction, amplitude, _ = _affine_fit(shape, y)
            scale = max(float(np.ptp(y)), float(np.max(np.abs(y))), 1e-15)
            chunk = (prediction - y) / scale
            if amplitude <= 0:
                chunk = np.r_[chunk, min(10.0, 1.0 - amplitude / scale)]
            chunks.append(chunk)
        return np.concatenate(chunks)

    result = least_squares(residual, initial, bounds=(lower, upper), method="trf",
                           loss="soft_l1", f_scale=0.02, max_nfev=2500)
    if not result.success:
        raise RuntimeError(f"Au {model} k={k} fit failed: {result.message}")
    rows = score_gold_model(traces, result.x, model, k)
    return result.x, rows


def score_gold_model(traces: Iterable[dict[str, object]], theta: np.ndarray,
                     model: str, k: int | None = None) -> list[dict[str, float]]:
    rows: list[dict[str, float]] = []
    for trace in traces:
        y = np.asarray(trace["current"], float)
        shape = _kinetic_shape(np.asarray(trace["time"], float), float(trace["bias"]),
                               theta, model, k)
        prediction, amplitude, offset = _affine_fit(shape, y)
        scale = max(float(np.ptp(y)), float(np.max(np.abs(y))), 1e-15)
        nrmse = float(np.sqrt(np.mean(np.square(prediction - y))) / scale)
        rows.append({
            "bias": float(trace["bias"]), "trial": int(trace["trial"]),
            "nrmse": nrmse, "amplitude_a": amplitude, "offset_a": offset,
        })
    return rows


def _bootstrap_mean_ci(values: np.ndarray, *, samples: int = 10000,
                       seed: int = 20260717) -> tuple[float, float]:
    values = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    means = np.empty(samples)
    for start in range(0, samples, 1000):
        stop = min(samples, start + 1000)
        draw = rng.integers(0, values.size, size=(stop - start, values.size))
        means[start:stop] = values[draw].mean(axis=1)
    return tuple(float(x) for x in np.percentile(means, (2.5, 97.5)))


def _bootstrap_grouped_mean_ci(values: np.ndarray, groups: np.ndarray, *,
                               samples: int = 10000,
                               seed: int = 20260717) -> tuple[float, float]:
    """Bootstrap a paired mean while resampling whole validation-bias groups.

    The 24 Au files are repeated runs from one device, not 24 independent devices.
    Trace-level resampling is retained as a descriptive sensitivity analysis, but the
    grouped interval is the conservative uncertainty statement for model comparison.
    """
    values = np.asarray(values, float)
    groups = np.asarray(groups)
    unique = np.unique(groups)
    group_means = np.asarray([values[groups == group].mean() for group in unique])
    return _bootstrap_mean_ci(group_means, samples=samples, seed=seed)


def identify_gold_models(gold_dir: Path, *, bootstrap_samples: int = 10000,
                         n_grid: int = 240) -> dict[str, object]:
    """Grouped leave-one-bias-out comparison of KWW and cascade models.

    The primary cascade representative minimises pooled held-out NRMSE. Fold votes
    are reported only as a stability diagnostic: using ``Counter.most_common`` made a
    tied vote depend on candidate ordering and previously froze an arbitrary ``k=1``.
    Exact stage count and support for a compressed rise are reported separately.
    """
    traces = load_gold_traces(gold_dir, n_grid=n_grid)
    training = [t for t in traces if float(t["bias"]) in TRAIN_BIASES]
    ood = [t for t in traces if float(t["bias"]) in OOD_BIASES]
    cv_rows: list[dict[str, object]] = []
    best_k_by_fold: list[int] = []
    kww_beta_by_fold: list[float] = []

    for heldout in TRAIN_BIASES:
        train_fold = [t for t in training if float(t["bias"]) != heldout]
        test_fold = [t for t in training if float(t["bias"]) == heldout]
        candidates: dict[str, list[dict[str, float]]] = {}
        theta, candidates["kww"] = _fit_gold_model(train_fold, "kww")
        kww_beta_by_fold.append(float(math.exp(theta[4])))
        candidates["kww"] = score_gold_model(test_fold, theta, "kww")
        for stages in range(1, 6):
            theta, _ = _fit_gold_model(train_fold, "cascade", stages)
            candidates[f"cascade_k{stages}"] = score_gold_model(
                test_fold, theta, "cascade", stages)
        best_k = min(range(1, 6), key=lambda stage: np.mean([
            row["nrmse"] for row in candidates[f"cascade_k{stage}"]
        ]))
        best_k_by_fold.append(best_k)
        for name, scored in candidates.items():
            for row in scored:
                cv_rows.append({"heldout_bias": heldout, "model": name, **row})

    full_parameters: dict[str, dict[str, float]] = {}
    full_scores: list[dict[str, object]] = []
    for name, model, stages in [
        ("kww", "kww", None),
        *[(f"cascade_k{k}", "cascade", k) for k in range(1, 6)],
    ]:
        theta, rows = _fit_gold_model(training, model, stages)
        params = {
            "tr0_s": math.exp(theta[0]), "cr_per_v": float(theta[1]),
            "td0_s": math.exp(theta[2]), "cd_per_v": float(theta[3]),
        }
        if model == "kww":
            params["beta_fill"] = math.exp(theta[4])
        full_parameters[name] = params
        for row in score_gold_model(ood, theta, model, stages):
            full_scores.append({"split": "ood", "model": name, **row})
        for row in rows:
            full_scores.append({"split": "training", "model": name, **row})

    def cv_values(name: str) -> np.ndarray:
        ordered = sorted((r for r in cv_rows if r["model"] == name),
                         key=lambda r: (r["bias"], r["trial"]))
        return np.asarray([r["nrmse"] for r in ordered], float)

    model_names = [f"cascade_k{stage}" for stage in range(1, 6)] + ["kww"]
    cv_mean_by_model = {
        name: float(cv_values(name).mean()) for name in model_names
    }
    selected_k = min(range(1, 6),
                     key=lambda stage: cv_mean_by_model[f"cascade_k{stage}"])
    winner_counts = {
        str(stage): int(best_k_by_fold.count(stage)) for stage in range(1, 6)
    }
    max_wins = max(winner_counts.values())
    modal_k = [int(stage) for stage, count in winner_counts.items()
               if count == max_wins]

    cascade = cv_values(f"cascade_k{selected_k}")
    mono = cv_values("cascade_k1")
    kww = cv_values("kww")
    improvement = mono - cascade
    heldout_groups = np.asarray([
        row["heldout_bias"] for row in sorted(
            (r for r in cv_rows if r["model"] == f"cascade_k{selected_k}"),
            key=lambda r: (r["bias"], r["trial"])
        )
    ])
    trace_improvement_ci = _bootstrap_mean_ci(
        improvement, samples=bootstrap_samples)
    grouped_improvement_ci = _bootstrap_grouped_mean_ci(
        improvement, heldout_groups, samples=bootstrap_samples)
    best_mean = cv_mean_by_model[f"cascade_k{selected_k}"]
    supported_k = [stage for stage in range(1, 6)
                   if cv_mean_by_model[f"cascade_k{stage}"] <= 1.05 * best_mean]
    fold_compression_consistent = bool(min(kww_beta_by_fold) > 1.0)
    exact_stage_resolved = bool(len(modal_k) == 1 and modal_k[0] == selected_k and
                                max_wins >= 4 and len(supported_k) == 1)
    evidence = {
        "selected_k": selected_k,
        "selection_rule": "minimum pooled grouped-LOBO trace-level mean NRMSE",
        "best_k_by_fold": best_k_by_fold,
        "fold_winner_counts": winner_counts,
        "modal_k": modal_k,
        "supported_k_within_5pct_of_best": supported_k,
        "exact_stage_count_status": "resolved" if exact_stage_resolved else "unresolved",
        "compressed_rise_status": (
            "consistent_across_bias_folds" if fold_compression_consistent
            else "not_consistent_across_bias_folds"
        ),
        "kww_beta_fill_by_fold": kww_beta_by_fold,
        "kww_beta_fill_full": full_parameters["kww"]["beta_fill"],
        "historical_beta_fill_reference": {
            "mean": 1.95, "sd": 0.30,
            "source": "historical manuscript averaged-trace analysis",
            "reconciliation_status": (
                "Both analyses support beta_fill > 1, but their numerical difference "
                "still requires a fit-window, preprocessing and objective audit."
            ),
        },
        "cv_mean_nrmse_by_model": cv_mean_by_model,
        "cascade_mean_nrmse": float(cascade.mean()),
        "single_stage_mean_nrmse": float(mono.mean()),
        "kww_mean_nrmse": float(kww.mean()),
        "selected_minus_single_stage_improvement_mean": float(improvement.mean()),
        "selected_minus_single_stage_trace_bootstrap_95ci": list(trace_improvement_ci),
        "selected_minus_single_stage_bias_bootstrap_95ci": list(grouped_improvement_ci),
        "selected_improves_single_stage_descriptively": bool(improvement.mean() > 0),
        "selected_improvement_resolved_by_bias_bootstrap": bool(
            grouped_improvement_ci[0] > 0),
        "selected_within_5pct_of_kww": bool(cascade.mean() <= 1.05 * kww.mean()),
        "interpretation": (
            "The pooled predictive representative is frozen for simulation; the "
            "compressed rise is consistent across bias folds, while the exact integer "
            "stage count remains a model-representation uncertainty."
        ),
    }
    return {
        "cv_rows": cv_rows, "fit_rows": full_scores, "parameters": full_parameters,
        "evidence": evidence, "train_biases": list(TRAIN_BIASES),
        "ood_biases": list(OOD_BIASES), "trace_count": len(traces),
        "measurement_unit": "one Au device; three recorded runs per bias",
        "validation_unit": "held-out bias; errors retained per recorded run",
    }


def _fit_ito_trace(time: np.ndarray, current: np.ndarray) -> dict[str, float]:
    valid = np.isfinite(time) & np.isfinite(current) & (current != 0)
    time, current = time[valid], np.abs(current[valid])
    peak_index = int(np.argmax(current))
    time, current = time[peak_index:] - time[peak_index], current[peak_index:]
    n_decay = int(time.size)
    if n_decay < 10:
        raise ValueError("fewer than 10 post-peak samples")
    if time.size > 300:
        indices = np.unique(np.linspace(0, time.size - 1, 300).astype(int))
        time, current = time[indices], current[indices]
    scale = float(np.max(current))
    y = current / max(scale, 1e-30)
    floor = float(np.median(y[-max(5, y.size // 10):]))
    amplitude = max(float(y[0] - floor), 0.05)

    def stretched(z: np.ndarray, beta_fixed: float | None = None) -> np.ndarray:
        amp, tau, floor_ = math.exp(z[0]), math.exp(z[1]), math.exp(z[-1])
        beta = beta_fixed if beta_fixed is not None else math.exp(z[2])
        return amp * np.exp(-np.power(time / tau, beta)) + floor_

    z0 = np.log([amplitude, max(float(np.median(time[time > 0])), 0.1), 0.6, max(floor, 1e-8)])
    lower = np.log([1e-5, 0.01, 0.1, 1e-8])
    upper = np.log([2.0, 1e4, 2.0, 2.0])
    result = least_squares(lambda z: stretched(z) - y, z0, bounds=(lower, upper),
                           loss="soft_l1", f_scale=0.03, max_nfev=3000)
    amp, tau, beta, floor_ = np.exp(result.x)
    fit = stretched(result.x)

    exp0 = np.log([amplitude, max(float(np.median(time[time > 0])), 0.1), max(floor, 1e-8)])
    exp_result = least_squares(
        lambda z: stretched(z, 1.0) - y, exp0,
        bounds=(np.log([1e-5, 0.01, 1e-8]), np.log([2.0, 1e4, 2.0])),
        loss="soft_l1", f_scale=0.03, max_nfev=3000,
    )
    exp_fit = stretched(exp_result.x, 1.0)
    total = float(np.sum(np.square(y - y.mean())))
    r2 = 1.0 - float(np.sum(np.square(y - fit))) / total if total > 0 else float("nan")
    r2_exp = 1.0 - float(np.sum(np.square(y - exp_fit))) / total if total > 0 else float("nan")
    rho = float(floor_ / max(amp + floor_, 1e-30))
    return {
        "peak_index": peak_index, "peak_fraction": peak_index / max(valid.sum(), 1),
        "n_decay": n_decay, "end_peak_ratio": float(current[-1] / max(current[0], 1e-30)),
        "amplitude_a": float(amp * scale), "floor_a": float(floor_ * scale),
        "tau_held_s": float(tau), "beta": float(beta), "rho": rho,
        "tau_fill_corrected_s": float(tau / max(1.0 - rho, 1e-6)),
        "r2_stretched": r2, "r2_exponential": r2_exp,
        "nrmse": float(np.sqrt(np.mean(np.square(y - fit)))),
    }


def _fit_ito_workbook(path: Path) -> dict[str, object] | None:
    """Spawn-safe fit and QC audit for one ITO workbook."""
    match = ITO_RE.search(path.name)
    protocol = match.group("protocol").lower().replace(" ", "_") if match else "unknown"
    if protocol != "voltage_bias":
        return None
    headers, values, metadata = _read_xls(path)
    try:
        time = values[:, headers.index("Time")]
        current = values[:, headers.index("BI")]
        voltage = values[:, headers.index("BV")]
        fit = _fit_ito_trace(time, current)
        reasons = []
        if fit["n_decay"] < 50:
            reasons.append("fewer_than_50_post_peak_samples")
        if fit["peak_fraction"] > 0.20:
            reasons.append("peak_after_first_20pct")
        if fit["end_peak_ratio"] > 0.80:
            reasons.append("less_than_20pct_observed_decay")
        if not (0.05 <= fit["tau_held_s"] <= 100.0):
            reasons.append("tau_outside_identifiable_range")
        if not (0.1 < fit["beta"] < 2.0):
            reasons.append("beta_on_fit_bound")
        if fit["r2_stretched"] < 0.80:
            reasons.append("r2_below_0.80")
        status = "included" if not reasons else "excluded"
    except Exception as exc:  # every failure remains in the audit table
        voltage = np.asarray([float(metadata.get("Bias", "nan"))])
        fit = {}
        status, reasons = "excluded", [f"fit_error:{type(exc).__name__}"]
    return {
        "filename": path.name, "run": int(match.group("run")) if match else "",
        "date": match.group("date") if match else metadata.get("Last Executed", ""),
        "bias_v": float(abs(np.nanmedian(voltage))), "qc_status": status,
        "qc_reason": "|".join(reasons), **fit,
    }


def analyse_ito(ito_dir: Path, *, bootstrap_samples: int = 10000,
                workers: int = 1) -> dict[str, object]:
    """Fit all voltage-bias workbooks and retain explicit QC outcomes.

    QC thresholds are observation-based and fixed here: at least 50 samples after
    the peak, peak in the first 20% of the record, at least a 20% observed decay,
    converged interior kinetic parameters, and R2 >= 0.80.  Beta is never used as
    an inclusion criterion, avoiding selection on the claimed dispersive outcome.
    """
    workbooks = sorted(ito_dir.glob("*.xls"))
    if int(workers) > 1:
        from multiprocessing import get_context
        with get_context("spawn").Pool(min(int(workers), len(workbooks))) as pool:
            fitted = pool.map(_fit_ito_workbook, workbooks, chunksize=1)
    else:
        fitted = map(_fit_ito_workbook, workbooks)
    rows = [row for row in fitted if row is not None]

    included = [row for row in rows if row["qc_status"] == "included"]
    if len(rows) != 70:
        raise ValueError(f"expected 70 voltage-bias ITO files, found {len(rows)}")
    if len(included) < 10:
        raise RuntimeError("too few valid ITO relaxations for population analysis")

    def included_at_r2(row: dict[str, object], threshold: float) -> bool:
        required = ("n_decay", "peak_fraction", "end_peak_ratio", "tau_held_s",
                    "beta", "r2_stretched")
        if any(key not in row for key in required):
            return False
        return bool(
            int(row["n_decay"]) >= 50 and
            float(row["peak_fraction"]) <= 0.20 and
            float(row["end_peak_ratio"]) <= 0.80 and
            0.05 <= float(row["tau_held_s"]) <= 100.0 and
            0.1 < float(row["beta"]) < 2.0 and
            float(row["r2_stretched"]) >= threshold
        )

    qc_sensitivity = []
    sensitivity_sets: dict[float, list[dict[str, object]]] = {}
    for r2_threshold in (0.70, 0.75, 0.80, 0.85, 0.90):
        sensitivity = [row for row in rows if included_at_r2(row, r2_threshold)]
        sensitivity_sets[r2_threshold] = sensitivity
        sensitivity_voltage = np.asarray([row["bias_v"] for row in sensitivity], float)
        sensitivity_tau = np.asarray([row["tau_held_s"] for row in sensitivity], float)
        sensitivity_design = np.column_stack((
            np.ones_like(sensitivity_voltage), np.sqrt(sensitivity_voltage)))
        sensitivity_coef, *_ = np.linalg.lstsq(
            sensitivity_design, np.log(1.0 / sensitivity_tau), rcond=None)
        qc_sensitivity.append({
            "r2_threshold": r2_threshold,
            "included_fits": len(sensitivity),
            "held_tau_median_s": float(np.median([
                row["tau_held_s"] for row in sensitivity
            ])),
            "beta_mean": float(np.mean([row["beta"] for row in sensitivity])),
            "beta_sd": float(np.std([row["beta"] for row in sensitivity], ddof=1)),
            "field_tau_zero_s": float(np.exp(-sensitivity_coef[0])),
            "field_slope_sqrt_v": float(sensitivity_coef[1]),
        })
    relaxed_53 = sensitivity_sets[0.70]
    primary_names = {str(row["filename"]) for row in included}
    added_at_070 = [str(row["filename"]) for row in relaxed_53
                    if str(row["filename"]) not in primary_names]
    voltage = np.asarray([row["bias_v"] for row in included], float)
    tau = np.asarray([row["tau_held_s"] for row in included], float)
    beta = np.asarray([row["beta"] for row in included], float)
    design = np.column_stack((np.ones_like(voltage), np.sqrt(voltage)))
    coef, *_ = np.linalg.lstsq(design, np.log(1.0 / tau), rcond=None)
    tau_zero = float(np.exp(-coef[0]))

    rng = np.random.default_rng(20260717)
    boot_tau0 = np.empty(bootstrap_samples)
    boot_slope = np.empty(bootstrap_samples)
    for start in range(0, bootstrap_samples, 1000):
        stop = min(bootstrap_samples, start + 1000)
        for j in range(start, stop):
            indices = rng.integers(0, len(included), size=len(included))
            xb = design[indices]
            cb, *_ = np.linalg.lstsq(xb, np.log(1.0 / tau[indices]), rcond=None)
            boot_tau0[j], boot_slope[j] = np.exp(-cb[0]), cb[1]
    quantiles = np.quantile(np.asarray([
        row["tau_fill_corrected_s"] for row in included
    ], float), (0.10, 0.25, 0.50, 0.75))
    summary = {
        "workbooks_total": 91, "voltage_bias_files": 70,
        "conditioning_files": 21, "included_fits": len(included),
        "excluded_fits": len(rows) - len(included), "experimental_unit": "workbook/run",
        "device_identity_available": False,
        "held_tau_median_s": float(np.median(tau)),
        "held_tau_range_s": [float(tau.min()), float(tau.max())],
        "beta_mean": float(beta.mean()), "beta_sd": float(beta.std(ddof=1)),
        "median_r2_stretched": float(np.median([r["r2_stretched"] for r in included])),
        "median_r2_exponential": float(np.median([r["r2_exponential"] for r in included])),
        "field_law": {
            "tau_zero_s": tau_zero, "slope_sqrt_v": float(coef[1]),
            "tau_zero_bootstrap_95ci_s": [float(x) for x in np.percentile(boot_tau0, (2.5, 97.5))],
            "slope_bootstrap_95ci": [float(x) for x in np.percentile(boot_slope, (2.5, 97.5))],
            "interpretation": "held-bias extrapolation; not a direct zero-field measurement",
        },
        "fill_corrected_tau_quantiles_s": {
            key: float(value) for key, value in zip(("q10", "q25", "q50", "q75"), quantiles)
        },
        "qc_sensitivity": {
            "primary_r2_threshold": 0.80,
            "rows": qc_sensitivity,
            "historical_count_53_reproduced_at_r2": 0.70,
            "additional_files_at_r2_0.70": added_at_070,
            "interpretation": (
                "Relaxing only the objective R2 threshold from 0.80 to 0.70 "
                "reproduces n=53 with negligible change in current summary kinetics. "
                "It does not by itself reproduce the historical beta mean, which "
                "still requires reconciliation of fitting and preprocessing."
            ),
        },
    }
    return {"rows": rows, "qc_sensitivity_rows": qc_sensitivity,
            "summary": summary}


def write_csv(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields: list[str] = []
    for row in rows:
        for key in row:
            if key not in fields:
                fields.append(key)
    with path.open("w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def write_json(path: Path, payload: object) -> None:
    def standards_safe(value):
        if isinstance(value, dict):
            return {str(key): standards_safe(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [standards_safe(item) for item in value]
        if isinstance(value, np.ndarray):
            return standards_safe(value.tolist())
        if isinstance(value, (np.floating, float)):
            number = float(value)
            return number if math.isfinite(number) else None
        if isinstance(value, (np.integer,)):
            return int(value)
        return value

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(standards_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )



In [ ]:
started = time.perf_counter()
gold_dir = resolve_gold_dir()
ito_dir = resolve_ito_dir()
raw_rows = raw_manifest(gold_dir, ito_dir)
gold_result = identify_gold_models(gold_dir, bootstrap_samples=BOOTSTRAP, n_grid=300)
# Workbook fits are intentionally serial here: notebook-local functions are not
# reliably pickleable under Windows/Jupyter spawn. The learning sweeps still use WORKERS.
ito_result = analyse_ito(ito_dir, bootstrap_samples=BOOTSTRAP, workers=1)
physical_seconds = time.perf_counter() - started

evidence = gold_result["evidence"]
selected_k = int(evidence["selected_k"])
supported_k = [int(k) for k in evidence["supported_k_within_5pct_of_best"]]
selected_law = gold_result["parameters"][f"cascade_k{selected_k}"]
quantile_map = ito_result["summary"]["fill_corrected_tau_quantiles_s"]
retention_quantiles = [float(quantile_map[key]) for key in ("q10", "q25", "q50", "q75")]
tau_r_09 = selected_law["tr0_s"] * math.exp(-selected_law["cr_per_v"] * 0.9)
tau_r_15 = selected_law["tr0_s"] * math.exp(-selected_law["cr_per_v"] * 1.5)

display({
    "raw files": len(raw_rows), "Au traces": gold_result["trace_count"],
    "selected k": selected_k, "supported k": supported_k,
    "stage-count status": evidence["exact_stage_count_status"],
    "candidate law": selected_law, "ITO summary": ito_result["summary"],
    "elapsed_s": physical_seconds,
})

model_names = list(evidence["cv_mean_nrmse_by_model"])
model_values = [evidence["cv_mean_nrmse_by_model"][name] for name in model_names]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(model_names, model_values, color="#3aa07a")
axes[0].tick_params(axis="x", rotation=45)
axes[0].set(ylabel="held-out NRMSE", title="Grouped Au model comparison")
axes[1].bar(["q10", "q25", "q50", "q75"], retention_quantiles, color="#2f4b8f")
axes[1].set(ylabel="fill-corrected retention (s)", title="Measured ITO quantiles")
fig.tight_layout()
plt.show()


## 2. Frozen prediction and learned preferred delay

The candidate physical law predicts the preferred delay before learning is evaluated. Device, decay-matched exponential and no-trace conditions share the same seeded task cells. Every physically supported cascade depth is repeated as a model-structure sensitivity analysis.


In [ ]:
from mrl_trace.selectivity import run_predictive_interval_sweep

if RUN_PROFILE == "publication":
    interval_seeds, interval_trials = 20, 1500
    sensitivity_seeds, sensitivity_trials = 20, 1500
    lag_factors = sensitivity_lags = (0.4, 0.7, 1.0, 1.4, 2.0)
elif RUN_PROFILE == "reduced":
    interval_seeds, interval_trials = 6, 500
    sensitivity_seeds, sensitivity_trials = 3, 300
    lag_factors, sensitivity_lags = (0.4, 0.7, 1.0, 1.4, 2.0), (0.7, 1.0, 1.4)
else:
    interval_seeds, interval_trials = 2, 80
    sensitivity_seeds, sensitivity_trials = 1, 30
    lag_factors, sensitivity_lags = (0.7, 1.0, 1.4), (1.0,)

started = time.perf_counter()
interval_result = run_predictive_interval_sweep(
    taus=retention_quantiles, seeds=interval_seeds, trials=interval_trials,
    lag_factors=lag_factors, k=selected_k, V=0.9,
    tau_r_override=tau_r_09, workers=WORKERS,
)
structure_results = {}
for stages in supported_k:
    law = gold_result["parameters"][f"cascade_k{stages}"]
    structure_tau_r = law["tr0_s"] * math.exp(-law["cr_per_v"] * 0.9)
    if RUN_PROFILE == "publication" and stages == selected_k:
        structure_results[stages] = interval_result
    else:
        structure_results[stages] = run_predictive_interval_sweep(
            taus=retention_quantiles, seeds=sensitivity_seeds,
            trials=sensitivity_trials, lag_factors=sensitivity_lags,
            k=stages, V=0.9, tau_r_override=structure_tau_r, workers=WORKERS,
        )
interval_seconds = time.perf_counter() - started

display({"primary": interval_result["diagnostics"],
         "by_k": {k: value["diagnostics"] for k, value in structure_results.items()},
         "elapsed_s": interval_seconds})

predicted = np.asarray([interval_result["predicted_tstar"][tau]
                        for tau in interval_result["taus"]], float)
learned = np.asarray([interval_result["learned_peak"][tau]
                      for tau in interval_result["taus"]], float)
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.plot(predicted, learned, "o-", color="#3aa07a")
limit = max(predicted.max(), learned.max())
ax.plot([0, limit], [0, limit], ":", color="0.3")
ax.set(xlabel="predicted preferred delay (s)", ylabel="learned preferred delay (s)",
       title="Material-to-learning timing prediction")
fig.tight_layout(); plt.show()


## 3. Existing design curve and new empirical-retention holdouts

The archived source tables already record the original 13-retention dense sweep. Here a predefined training grid supplies a task-specific design curve, and the four measured ITO quantiles are evaluated without refitting.


In [ ]:
from mrl_trace.maze import run_dmax_adaptive

if RUN_PROFILE == "publication":
    horizon_seeds, horizon_episodes = 20, 800
    delay_ratios, training_taus = (3, 5, 7, 9, 12, 16, 20), (1., 2., 5., 10., 20., 40.)
elif RUN_PROFILE == "reduced":
    horizon_seeds, horizon_episodes = 6, 300
    delay_ratios, training_taus = (4, 8, 12, 16, 20), (1., 2., 5., 10., 20.)
else:
    horizon_seeds, horizon_episodes = 2, 80
    delay_ratios, training_taus = (4, 8, 12, 16, 20), (1., 2., 5., 10.)

started = time.perf_counter()
horizon_training = run_dmax_adaptive(
    seeds=horizon_seeds, episodes=horizon_episodes, taus=training_taus,
    delay_ratios=delay_ratios, workers=WORKERS, device_k=selected_k,
    tau_r_override=tau_r_15,
)
holdout_taus = sorted(set(round(tau, 4) for tau in retention_quantiles))
horizon_holdout = run_dmax_adaptive(
    seeds=horizon_seeds, episodes=horizon_episodes, taus=holdout_taus,
    delay_ratios=delay_ratios, workers=WORKERS, device_k=selected_k,
    tau_r_override=tau_r_15,
)
x = np.asarray(horizon_training["taus"], float)
y = np.asarray([horizon_training["dmax"][tau] for tau in horizon_training["taus"]], float)
slope = float(np.sum(x * y) / np.sum(x * x))
residual_sd = float(np.std(y - slope * x, ddof=1)) if len(x) > 1 else 0.0
hx = np.asarray(horizon_holdout["taus"], float)
observed = np.asarray([horizon_holdout["dmax"][tau] for tau in horizon_holdout["taus"]], float)
prediction = slope * hx
half_width = 1.96 * residual_sd * np.sqrt(1.0 + hx * hx / np.sum(x * x))
covered = np.abs(observed - prediction) <= half_width
resolved = observed > 0
relative_error = np.full(observed.shape, np.nan)
relative_error[resolved] = np.abs(observed[resolved] - prediction[resolved]) / observed[resolved]
horizon_diagnostics = {
    "holdouts": len(hx), "covered": int(covered.sum()),
    "median_absolute_relative_error": float(np.median(relative_error[resolved])) if resolved.any() else None,
    "zero_or_unresolved_holdouts": int((~resolved).sum()),
}
horizon_seconds = time.perf_counter() - started
display({"diagnostics": horizon_diagnostics, "elapsed_s": horizon_seconds})

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(x, y, "o", color="#2f4b8f", label="predefined training grid")
ax.plot(hx, prediction, "--", color="0.4", label="frozen prediction")
ax.errorbar(hx, observed, yerr=half_width, fmt="s", color="#3aa07a", label="ITO holdouts")
ax.set(xlabel="retention (s)", ylabel="maximum learned delay (s)", title="Held-out design curve")
ax.legend(frameon=False); fig.tight_layout(); plt.show()


## 4. Retention benefit and flexibility cost

The existing experiment record contains reversal at one retention. This extension runs device and decay-matched exponential traces across all four empirical-retention quantiles, retaining censored recovery times rather than selecting the largest contrast.


In [ ]:
from mrl_trace.bandit import run_reversal, reversal_phase_final, reversal_recriterion

if RUN_PROFILE == "publication":
    reversal_seeds, reversal_trials, dt, cue_dur, reward_delay = 20, 3000, 0.005, 1.0, 2.0
elif RUN_PROFILE == "reduced":
    reversal_seeds, reversal_trials, dt, cue_dur, reward_delay = 6, 1200, 0.01, 0.6, 1.0
else:
    reversal_seeds, reversal_trials, dt, cue_dur, reward_delay = 3, 400, 0.02, 0.4, 0.6

started = time.perf_counter()
reversal_rows = []
reversal_raw = {}
for tau in retention_quantiles:
    for condition in ("device", "abstract"):
        rewards, flips = run_reversal(
            condition, B=reversal_seeds, tau_leak=float(tau), trials=reversal_trials,
            dt=dt, cue_dur=cue_dur, D=reward_delay, device_k=selected_k,
            tau_r_override=tau_r_09,
        )
        reversal_raw[(float(tau), condition)] = (rewards, flips)
        window = min(200, reversal_trials // 5)
        pre, post = reversal_phase_final(rewards, flips, reversal_trials, window=window)
        criterion_window = min(100, max(20, reversal_trials // 10))
        for seed in range(reversal_seeds):
            recovery = reversal_recriterion(rewards[seed], flips[0], 0.75, window=criterion_window)
            reversal_rows.append({
                "tau_s": float(tau), "condition": condition, "seed": seed,
                "pre_rate": float(pre[seed]), "post_rate": float(post[seed]),
                "recovery_trials": int(recovery) if recovery is not None else reversal_trials // 2,
                "recovery_censored": recovery is None,
            })
reversal_seconds = time.perf_counter() - started
reversal_summary = []
for tau in retention_quantiles:
    for condition in ("device", "abstract"):
        rows = [row for row in reversal_rows if row["tau_s"] == float(tau) and row["condition"] == condition]
        reversal_summary.append({
            "tau_s": float(tau), "condition": condition,
            "pre_rate": float(np.mean([row["pre_rate"] for row in rows])),
            "post_rate": float(np.mean([row["post_rate"] for row in rows])),
            "recovery_trials": float(np.mean([row["recovery_trials"] for row in rows])),
            "censored_fraction": float(np.mean([row["recovery_censored"] for row in rows])),
        })
display({"summary": reversal_summary, "elapsed_s": reversal_seconds})

fig, ax = plt.subplots(figsize=(5, 3.5))
for condition, colour in (("device", "#3aa07a"), ("abstract", "#2f4b8f")):
    rows = [row for row in reversal_summary if row["condition"] == condition]
    ax.plot([row["tau_s"] for row in rows], [row["recovery_trials"] for row in rows],
            "o-", color=colour, label=condition)
ax.set(xlabel="retention (s)", ylabel="recovery trials", title="Retention--flexibility trade-off")
ax.legend(frameon=False); fig.tight_layout(); plt.show()


## Result archive and interpretation

No result is written by default. When saving is enabled, the complete Python result objects and a compact JSON report are placed in the chosen output directory, which should be archived with the environment, commit and execution log. Publication-profile output belongs in the DOI deposit, not the Git repository.

Smoke and reduced results are feasibility evidence only. The notebook reports scientific diagnostics and never turns them into a journal acceptance decision.


In [ ]:
def jsonable(value):
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, (np.floating, float)):
        number = float(value); return number if math.isfinite(number) else None
    if isinstance(value, np.integer): return int(value)
    if isinstance(value, dict): return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)): return [jsonable(item) for item in value]
    return value

run_report = {
    "profile": RUN_PROFILE, "bootstrap": BOOTSTRAP, "workers": WORKERS,
    "physical_seconds": physical_seconds, "interval_seconds": interval_seconds,
    "horizon_seconds": horizon_seconds, "reversal_seconds": reversal_seconds,
    "selected_k": selected_k, "supported_k": supported_k,
    "candidate_law": selected_law, "retention_quantiles_s": retention_quantiles,
    "physical_evidence": evidence, "interval_diagnostics": interval_result["diagnostics"],
    "horizon_diagnostics": horizon_diagnostics, "reversal_summary": reversal_summary,
    "evidential_use": "publication-scale" if RUN_PROFILE == "publication" else "feasibility-only",
    "editorial_decision": "not_assessed", "provenance": PROVENANCE_RECORD,
}
display(run_report)

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    payload = {
        "run_report": run_report, "raw_manifest": raw_rows, "gold": gold_result,
        "ito": ito_result, "interval": interval_result, "structure": structure_results,
        "horizon_training": horizon_training, "horizon_holdout": horizon_holdout,
        "reversal_rows": reversal_rows,
    }
    np.save(OUTPUT_DIR / f"nmi_predictive_linkage_{RUN_PROFILE}.npy", payload, allow_pickle=True)
    (OUTPUT_DIR / f"nmi_predictive_linkage_{RUN_PROFILE}.json").write_text(
        json.dumps(jsonable(run_report), indent=2, allow_nan=False) + "\n", encoding="utf-8")
    print(f"saved result archive to {OUTPUT_DIR}")
